In [1]:
pip install torch transformers datasets scikit-learn numpy

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from transformers import BertTokenizer, BertModel
from datasets import load_dataset

import numpy as np
from sklearn.metrics.pairwise import  cosine_similarity
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)



cuda


In [3]:
#Hyperparams

EMBED_DIM = 768
BATCH_SIZE = 32
EPOCHS = 3
LR = 1e-5
TEMPERATURE = 0.05
MAX_LEN = 128

In [4]:
dataset = load_dataset("snli", split='train[:10000]')

README.md: 0.00B [00:00, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/412k [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/413k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/19.6M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/550152 [00:00<?, ? examples/s]

In [5]:
dataset

Dataset({
    features: ['premise', 'hypothesis', 'label'],
    num_rows: 10000
})

In [6]:
dataset[0]

{'premise': 'A person on a horse jumps over a broken down airplane.',
 'hypothesis': 'A person is training his horse for a competition.',
 'label': 1}

In [7]:
dataset[2]

{'premise': 'A person on a horse jumps over a broken down airplane.',
 'hypothesis': 'A person is outdoors, on a horse.',
 'label': 0}

In [8]:
triplets = []

for ex in tqdm(dataset, desc="processing SNLI"):
  if ex['label'] == 0 or ex['label'] == 2:
    premise = ex['premise']
    hypothesis = ex['hypothesis']
    label = ex['label']


premise = [ex['premise'] for ex in dataset if ex['label'] in [0,2]]
positives = [ex['hypothesis'] for ex in dataset if ex['label'] == 0]
negatives = [ex['hypothesis'] for ex in dataset if ex['label'] == 2]




processing SNLI: 100%|██████████| 10000/10000 [00:00<00:00, 15993.05it/s]


In [9]:
premise

['A person on a horse jumps over a broken down airplane.',
 'A person on a horse jumps over a broken down airplane.',
 'Children smiling and waving at camera',
 'Children smiling and waving at camera',
 'A boy is jumping on skateboard in the middle of a red bridge.',
 'A boy is jumping on skateboard in the middle of a red bridge.',
 'An older man sits with his orange juice at a small table in a coffee shop while employees in bright colored shirts smile in the background.',
 'Two blond women are hugging one another.',
 'Two blond women are hugging one another.',
 'A few people in a restaurant setting, one of them is drinking orange juice.',
 'A few people in a restaurant setting, one of them is drinking orange juice.',
 'An older man is drinking orange juice at a restaurant.',
 'An older man is drinking orange juice at a restaurant.',
 'A man with blond-hair, and a brown shirt drinking out of a public water fountain.',
 'A man with blond-hair, and a brown shirt drinking out of a public 

In [10]:
positives

['A person is outdoors, on a horse.',
 'There are children present',
 'The boy does a skateboarding trick.',
 'There are women showing affection.',
 'The diners are at a restaurant.',
 'A man is drinking juice.',
 'A blond man drinking water from a fountain.',
 'There are two woman in this picture.',
 'Two women hug each other.',
 'A team is trying to tag a runner out.',
 'A school is hosting an event.',
 'Women are waiting by a tram.',
 'A family of three is at the beach.',
 'There are people just getting on a train',
 'There are people waiting on a train.',
 'A couple are playing with a young child outside.',
 'The family is outside.',
 'Near a couple of restaurants, two people walk across the street.',
 'The woman is wearing green.',
 'A woman in white.',
 'A man is advertising for a restaurant.',
 'The woman is wearing white.',
 'They are walking with a sign.',
 'The woman and man are outdoors.',
 'The adults are both male and female.',
 'Two adults walk across a street.',
 'Two pe

In [11]:
negatives

['A person is at a diner, ordering an omelette.',
 'The kids are frowning',
 'The boy skates down the sidewalk.',
 'A boy flips a burger.',
 'The women are sleeping.',
 'The people are sitting at desks in school.',
 'Two women are at a restaurant drinking wine.',
 'A blond man wearing a brown shirt is reading a book on a bench in the park',
 'The friends scowl at each other over a full dinner table.',
 'Two groups of rival gang members flipped each other off.',
 'A team is playing baseball on Saturn.',
 'A school hosts a basketball game.',
 'The women do not care what clothes they wear.',
 'A family of three is at the mall shopping.',
 'A couple watch a little girl play by herself on the beach.',
 'The family is sitting down for dinner.',
 'The people are standing still on the curb.',
 'The woman is nake.',
 'They are protesting outside the capital.',
 'The woman is wearing black.',
 'Olympic swimming.',
 "A man and a soman are eating together at John's Pizza and Gyro.",
 "The man is s

In [12]:
min_len = min(len(premise), len(positives), len(negatives))

In [13]:
min_len

3329

In [14]:
premise = premise[:min_len]
positives = positives[:min_len]
negatives = negatives[:min_len]

In [15]:
triplets = list(zip(premise, positives, negatives))

In [16]:
len(triplets)

3329

In [17]:
triplets[0]

('A person on a horse jumps over a broken down airplane.',
 'A person is outdoors, on a horse.',
 'A person is at a diner, ordering an omelette.')

In [18]:
class TripletDataset(Dataset):
    def __init__(self, triplets, tokenizer, max_len=128):
        self.triplets = triplets
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        anchor, positive, negative = self.triplets[idx]

        anchor_enc = self.tokenizer(
            anchor,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        positive_enc = self.tokenizer(
            positive,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        negative_enc = self.tokenizer(
            negative,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        # remove the extra batch dimension (1, max_len) → (max_len,)
        return {
            'anchor': {k: v.squeeze(0) for k, v in anchor_enc.items()},
            'positive': {k: v.squeeze(0) for k, v in positive_enc.items()},
            'negative': {k: v.squeeze(0) for k, v in negative_enc.items()},
        }

# Custom collate function so DataLoader merges dicts cleanly
def collate_fn(batch):
    keys = ['anchor', 'positive', 'negative']
    output = {}

    for key in keys:
        input_ids = torch.stack([item[key]['input_ids'] for item in batch])
        attention_mask = torch.stack([item[key]['attention_mask'] for item in batch])
        output[key] = {'input_ids': input_ids, 'attention_mask': attention_mask}

    return output

In [19]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
dataset = TripletDataset(triplets, tokenizer)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [20]:
class SimCSEModel(nn.Module):
    def __init__(self, model_name='bert-base-uncased', embed_dim=768):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.pooler = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        pooled = self.dropout(pooled)
        pooled = self.pooler(pooled)
        return F.normalize(pooled, p=2, dim=1)

model = SimCSEModel().to(device)
print(model)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

SimCSEModel(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

In [21]:
def contrastive_loss(anchors, positives, negatives, temperature=TEMPERATURE):
    all_embeds = torch.cat([anchors, positives, negatives], dim=0)
    sim_matrix = torch.mm(anchors, all_embeds.T) / temperature
    batch_size = anchors.size(0)
    labels = torch.arange(batch_size, device=anchors.device) + batch_size
    return F.cross_entropy(sim_matrix, labels)

In [22]:
dummy_emb = torch.randn(2, EMBED_DIM).to(device)
print("Loss test: ", contrastive_loss(dummy_emb, dummy_emb, dummy_emb).item())

Loss test:  1.0986123085021973


In [23]:
model = SimCSEModel().to(device)
optimizer = optim.AdamW(model.parameters(), lr=LR)

model.train()
total_losses = []

for epoch in range(EPOCHS):
    epoch_loss = 0
    num_batches = 0

    for batch in tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        optimizer.zero_grad()

        anchor_ids = batch['anchor']['input_ids'].to(device)
        anchor_mask = batch['anchor']['attention_mask'].to(device)
        pos_ids = batch['positive']['input_ids'].to(device)
        pos_mask = batch['positive']['attention_mask'].to(device)
        neg_ids = batch['negative']['input_ids'].to(device)
        neg_mask = batch['negative']['attention_mask'].to(device)

        anchor_emb = model(anchor_ids, anchor_mask)
        pos_emb = model(pos_ids, pos_mask)
        neg_emb = model(neg_ids, neg_mask)

        loss = contrastive_loss(anchor_emb, pos_emb, neg_emb)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    avg_loss = epoch_loss / num_batches
    total_losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{EPOCHS}, Average Loss: {avg_loss:.4f}")

print("✅ Training Complete!")

Epoch 1/3: 100%|██████████| 105/105 [03:08<00:00,  1.80s/it]


Epoch 1/3, Average Loss: 4.9108


Epoch 2/3: 100%|██████████| 105/105 [03:09<00:00,  1.81s/it]


Epoch 2/3, Average Loss: 4.5782


Epoch 3/3: 100%|██████████| 105/105 [03:09<00:00,  1.81s/it]

Epoch 3/3, Average Loss: 4.5535
✅ Training Complete!


In [24]:
model.eval()


SimCSEModel(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

In [25]:
text = "AI agents can automate complex workflows."

enc = tokenizer(
    text,
    max_length=128,
    padding='max_length',
    truncation=True,
    return_tensors='pt'
).to(device)


In [26]:
with torch.no_grad():
    embedding = model(enc['input_ids'], enc['attention_mask'])


In [27]:
embedding = F.normalize(embedding, p=2, dim=1)  # if not already normalized
embedding_np = embedding.cpu().numpy()


In [28]:
def get_embedding(model, tokenizer, text, device, max_len=128):
    model.eval()
    enc = tokenizer(
        text,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    ).to(device)

    with torch.no_grad():
        emb = model(enc['input_ids'], enc['attention_mask'])
    return emb.squeeze(0).cpu().numpy()


In [29]:
embedding = get_embedding(model, tokenizer, "AI is transforming industries.", device)
print(embedding.shape)  # (768,)


(768,)


In [30]:
embedding

array([ 2.65968181e-02, -4.89783809e-02, -3.26208025e-02,  4.30249311e-02,
        3.73437516e-02, -7.57481456e-02, -4.59684581e-02, -9.85847786e-03,
       -2.50532739e-02,  2.06567589e-02, -1.04579646e-02, -2.52436493e-02,
        1.59741566e-02,  1.13841761e-02,  5.63366041e-02,  4.09920840e-03,
       -6.14156062e-03, -8.18841439e-03,  4.78572957e-02, -1.53427478e-03,
        2.75464989e-02,  1.24825286e-02, -8.41164961e-02, -1.04118635e-05,
       -6.05740398e-02,  2.80803982e-02, -2.97931358e-02, -3.10183503e-02,
        9.99161880e-03,  2.59487908e-02,  6.73369924e-03,  5.57358265e-02,
       -7.36539997e-03,  2.49941293e-02, -4.81196381e-02,  4.45595868e-02,
        1.29080126e-02,  3.36108096e-02,  1.24774296e-02,  2.33422797e-02,
       -3.27171907e-02, -2.65243743e-02, -1.78633034e-02,  3.03101446e-02,
       -4.63870093e-02,  4.94054577e-04, -3.01025566e-02,  9.07206349e-03,
        5.85849583e-02,  3.73151340e-02, -5.68976030e-02, -5.28269932e-02,
       -8.56735185e-03, -

In [37]:
import torch
import pandas as pd
from datasets import load_dataset
from sklearn.metrics.pairwise import cosine_similarity

# Load STS dataset
sts = load_dataset("stsb_multi_mt", name="en", split="test[:100]")

# Convert to DataFrame
sts = pd.DataFrame(sts)

# Define embedding function
def embed_texts(texts):
    if isinstance(texts, pd.Series):
        texts = texts.tolist()  # ✅ convert Series to list
    enc = tokenizer(texts, padding=True, truncation=True, return_tensors='pt').to('cuda')
    with torch.no_grad():
        return model(enc['input_ids'], enc['attention_mask']).cpu()

# Generate embeddings
emb1 = embed_texts(sts['sentence1'])
emb2 = embed_texts(sts['sentence2'])

# Compute cosine similarity
similarities = cosine_similarity(emb1, emb2).diagonal()

# Evaluate correlation with human labels
from scipy.stats import spearmanr
corr = spearmanr(similarities, sts['similarity_score'])
print("Spearman correlation:", corr.correlation)


Spearman correlation: 0.1842270746848136
